In [1]:
import sys
# add path to gp-quadrature
sys.path.append('/Users/colecitrenbaum/Documents/GPs/gp-quadrature')
sys.path.append('/Users/colecitrenbaum/Documents/GPs/gp-quadrature/kernels')
from kernels import SquaredExponential
from vanilla_gp_sampling import sample_bernoulli_gp
from efgpnd import ToeplitzND, compute_convolution_vector_vectorized_dD

import torch
from efgpnd import NUFFT
from cg import ConjugateGradients
from torch import vmap
#default to float64
torch.set_default_dtype(torch.float64)

# --- Parameters ---
n = 1000  # Number of points
d = 1  # Dimensionality of the input space
true_length_scale =0.6
true_variance = 1
dtype = torch.float64  # Use float64 as in the original example
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")  # Use GPU if available
print(f"Using device: {device}")

# --- Generate Input Points ---
# Generate random points in d-dimensional space from -1 to 1
x = torch.rand(n, d, dtype=dtype, device=device) * 2 - 1
y,f = sample_bernoulli_gp(x,length_scale=true_length_scale,variance=true_variance)
rdtype = torch.float64
cdtype = torch.complex128

import torch
import torch.nn as nn
from typing import List, Iterator, Tuple, Optional, Union
from torch.optim import Adam
from utils.kernels import get_xis
import math

class qVariationalParams(nn.Module): # inherits from nn.Module
    def __init__(self):
        """

        """
        super().__init__()
        self.Delta = nn.Parameter(torch.full((n,), .2, dtype=torch.float64))

Using device: cpu
Cholesky succeeded after adding jitter.


In [17]:
q = qVariationalParams()
cdtype = torch.complex128
rdtype = torch.float64
true_length_scale = 0.4
true_variance = 1
kernel = SquaredExponential(dimension = d,init_lengthscale=0.5*true_length_scale,init_variance=5*true_variance)
eps = 1e-4


dtype = x.dtype
rdtype = dtype 
x       = x.to(device, dtype)
y       = y.to(device, dtype)
if x.ndim == 1:
    x = x.unsqueeze(-1)
x0 = x.min(dim=0).values  
x1 = x.max(dim=0).values  

if x.ndim == 1:
    x = x.unsqueeze(-1)
d = x.shape[1]
domain_lengths = x1 - x0
L = domain_lengths.max()
N = x.shape[0]
xis_1d, h, mtot = get_xis(kernel_obj=kernel, eps=eps, L=L, use_integral=True, l2scaled=False)
grids = torch.meshgrid(*(xis_1d for _ in range(d)), indexing='ij') # makes tensor product Jm 
xis = torch.stack(grids, dim=-1).view(-1, d) 
ws2 = kernel.spectral_density(xis).to(dtype=cdtype) * h**d
ws = torch.sqrt(ws2) # (mtot**d,1)
logws = kernel.log_spectral_density(xis).to(dtype=cdtype) + 1/2*math.log(h**d)

m_conv = (mtot - 1) // 2
v_kernel = compute_convolution_vector_vectorized_dD(m_conv, x, h).to(dtype=cdtype)
toeplitz = ToeplitzND(v_kernel, force_pow2=True)   
Dprime  = (h**d * kernel.spectral_grad(xis)).to(cdtype)  # (M, 3)
delta = (q.Delta.to(dtype=cdtype, device=device)).clone()

# Sigma_z_batch = make_Sigma_z_batch(delta)

def naive_kernel(x,xis):
    F_train = torch.exp(2 * math.pi * 1j * torch.matmul(x, xis.T)).to(cdtype)
    return F_train
F_train = naive_kernel(x,xis)


# Create the NUFFT operator
nufft_eps = 1e-15
OUT = (mtot,)*d
nufft_op = NUFFT(x, torch.zeros_like(x), h, nufft_eps, cdtype=cdtype, device=device)

# Define the simplified helper functions
fadj = lambda v: nufft_op.type1(v, out_shape=OUT).reshape(-1)    # F* apply: nonuniform → uniform
fwd = lambda fk: nufft_op.type2(fk, out_shape=OUT)                # F apply:  uniform → nonuniform

In [19]:
# 1) compute log(D2)
logD2 = kernel.log_spectral_density(xis)         # log spectral_density(xis)
logD2 = logD2 + xis.new_tensor(d) * torch.log(h) # add d·log h

# 2) invert
invD2 = torch.exp(-logD2)                        # 1/D2


In [18]:
torch.exp(-2*logws) 

tensor([4.7602e+07-0.j, 1.4316e+06-0.j, 6.2259e+04-0.j, 3.9154e+03-0.j, 3.5608e+02-0.j,
        4.6827e+01-0.j, 8.9051e+00-0.j, 2.4489e+00-0.j, 9.7387e-01+0.j, 5.6003e-01+0.j,
        4.6571e-01+0.j, 5.6003e-01+0.j, 9.7387e-01+0.j, 2.4489e+00-0.j, 8.9051e+00-0.j,
        4.6827e+01-0.j, 3.5608e+02-0.j, 3.9154e+03-0.j, 6.2259e+04-0.j, 1.4316e+06-0.j,
        4.7602e+07-0.j])

In [3]:
jitter_val = 1e-8
v_kernel = compute_convolution_vector_vectorized_dD(m_conv, x, h).to(dtype=cdtype)
toeplitz = ToeplitzND(v_kernel, force_pow2=True)
A_apply = lambda x: toeplitz(x) + jitter_val * x
fadj_batched = vmap(fadj, in_dims=0, out_dims=0)
z_test = torch.randn(n, device=device)
def D2_Fstar_Kinv_z(z):

    rhs = fadj(z)
    x0 = torch.zeros_like(rhs)
    return ConjugateGradients(
        A_apply, rhs, x0,
        tol=1e-20, early_stopping=False, max_iter = 10000
    ).solve(), rhs
out,_ = D2_Fstar_Kinv_z(z_test.flatten())

print(out/ws2)
F_train = naive_kernel(x, xis)
jitter = 1e-8
W_F_train_star = torch.diag(ws) @ F_train.conj().T
K_train = (F_train * ws) @ W_F_train_star
Kinv = torch.linalg.inv(K_train + jitter * torch.eye(n, device=device))
D2 = torch.diag(ws2)
print( (F_train.conj().T @ Kinv@ z_test.to(dtype=cdtype)).flatten())

tensor([-1782.1173-8.5584e+02j,  1498.7935+5.2447e+02j,  -950.1128-2.8297e+02j,
          553.5660+1.3646e+02j,  -325.3398-6.3533e+01j,   200.1077+3.0457e+01j,
         -131.4714-1.5368e+01j,    93.6740+7.9505e+00j,   -73.1581-4.0867e+00j,
           62.9610+1.7658e+00j,   -59.8762-1.5705e-08j,    62.9610-1.7658e+00j,
          -73.1581+4.0867e+00j,    93.6740-7.9505e+00j,  -131.4714+1.5368e+01j,
          200.1077-3.0457e+01j,  -325.3398+6.3533e+01j,   553.5660-1.3646e+02j,
         -950.1128+2.8297e+02j,  1498.7935-5.2447e+02j, -1782.1173+8.5584e+02j])
tensor([-1778.2322-8.5563e+02j,  1495.4099+5.2431e+02j,  -947.9280-2.8288e+02j,
          552.2653+1.3641e+02j,  -324.5652-6.3508e+01j,   199.6270+3.0444e+01j,
         -131.1530-1.5362e+01j,    93.4457+7.9470e+00j,   -72.9791-4.0849e+00j,
           62.8066+1.7650e+00j,   -59.7293-3.2242e-05j,    62.8066-1.7650e+00j,
          -72.9791+4.0848e+00j,    93.4457-7.9469e+00j,  -131.1530+1.5362e+01j,
          199.6270-3.0444e+01j,  -324.5